[Guide Index](README.md)


# SPARQL Entailment — User Guide

Inferencing is the process of deriving new information from existing graph information according to a specified set of rules, semantics, or expressions.

As described in the [Graph Inferencing guide](02b-graphs-inferencing.ipynb), a reasoning *entailment regime* can be applied to a graph directly via `infer()`. The resultant graph can then be queried as described in the [SPARQL guide](03-sparql.ipynb).

Alternatively, `StarLayerGraph.query()` lets a query itself specify the *entailment regime* to apply via `entailment=`, without first running inference on the underlying graph. This guide is about that query-time mechanism specifically. The entailment regimes themselves - what each one means, the full regime-support status table, and which one to pick - are explained once, in full, in the [Graph Inferencing guide](02b-graphs-inferencing.ipynb); this guide just shows the same regimes applied via `query(entailment=...)` instead of `infer()`.


## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01-getting-started.ipynb) if not. Run cells from top to bottom; later sections may reuse variables from earlier sections.


In [1]:
from starlayer.graph import ENTAILMENT, StarLayerGraph, Namespace, RDF

EX = Namespace("http://example.org/")

## 1. `entailment=ENTAILMENT.RDF` — RDF Entailment

The narrowest regime - see the [Graph Inferencing guide](02b-graphs-inferencing.ipynb) for what it covers. Query-time only: there's no `infer()` form, since `infer()` doesn't support `ENTAILMENT.RDF`.


In [2]:
g4 = StarLayerGraph()
g4.bind("ex", EX)
g4.parse(data='''
    @prefix ex: <http://example.org/> .
    ex:alice ex:worksAt ex:Acme .
''', format="turtle12")

QUERY_RDF = "PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>  SELECT ?p WHERE { ?p a rdf:Property }"

print("rdf:Property in the graph:")
print("plain (no entailment):", [str(r.p) for r in g4.query(QUERY_RDF)])
print("entailment=ENTAILMENT.RDF:       ", [str(r.p) for r in g4.query(QUERY_RDF, entailment=ENTAILMENT.RDF)])

rdf:Property in the graph:
plain (no entailment): []
entailment=ENTAILMENT.RDF:        ['http://example.org/worksAt']


## 2. `entailment=ENTAILMENT.RDFS` — RDFS Entailment

See the [Graph Inferencing guide](02b-graphs-inferencing.ipynb#2.-RDFS-reasoning) for what the RDFS ruleset covers (`rdfs:subClassOf`/`rdfs:domain`/`rdfs:range`/`rdfs:subPropertyOf`). Applied here at query time: `entailment=ENTAILMENT.RDFS` queries are handled through a query-time rewrite of the SPARQL query against the current graph data - no copy or mutation of the original graph is required.


In [3]:
# subclass entailment.  Alice is an employee entails from the fact she is a manager.
g = StarLayerGraph()
g.bind("ex", EX)
g.parse(data='''
    @prefix ex: <http://example.org/> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    ex:Manager rdfs:subClassOf ex:Employee .
    ex:Employee rdfs:subClassOf ex:Person .
    ex:alice a ex:Manager .
''', format="turtle12")

QUERY = "PREFIX ex: <http://example.org/> SELECT ?x WHERE { ?x a ex:Employee }"

print("plain (no entailment):", [str(r.x) for r in g.query(QUERY)])
print("entailment=ENTAILMENT.RDFS:       ", [str(r.x) for r in g.query(QUERY, entailment=ENTAILMENT.RDFS)])
print("triple count unchanged - no data was copied or added:", len(g))

plain (no entailment): []
entailment=ENTAILMENT.RDFS:        ['http://example.org/alice']
triple count unchanged - no data was copied or added: 3


### 2.1 Reflexivity and multi-hop transitivity

`rdfs:subClassOf` entailment, applied at query time:


In [4]:
# Alice is a person is entailed from the fact that she is an employee which is entailed from the fact she is a manager.
rows = g.query("PREFIX ex: <http://example.org/> SELECT ?type WHERE { ex:alice a ?type }", entailment=ENTAILMENT.RDFS)
print("every entailed type:", sorted(str(r.type) for r in rows))

rows = g.query("PREFIX ex: <http://example.org/> SELECT ?x WHERE { ?x a ex:Person }", entailment=ENTAILMENT.RDFS)
print("two hops away (Manager -> Employee -> Person):", [str(r.x) for r in rows])

every entailed type: ['http://example.org/Employee', 'http://example.org/Manager', 'http://example.org/Person']
two hops away (Manager -> Employee -> Person): ['http://example.org/alice']


### 2.2 `rdfs:domain`, `rdfs:range`, and `rdfs:subPropertyOf`

The rest of the RDFS "data" ruleset, also applied at query time:


In [5]:
# If we know that Alice works at Acme that entails Alice is a person and Acme is an organization.
# If we know that Alice has a parent bob, that entails that Alice has a relative bob.
g2 = StarLayerGraph()
g2.bind("ex", EX)
g2.parse(data='''
    @prefix ex: <http://example.org/> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    ex:worksAt rdfs:domain ex:Person ;
               rdfs:range ex:Organization .
    ex:hasParent rdfs:subPropertyOf ex:hasRelative .
    ex:alice ex:worksAt ex:Acme .
    ex:alice ex:hasParent ex:bob .
''', format="turtle12")

rows = g2.query("PREFIX ex: <http://example.org/> SELECT ?x WHERE { ?x a ex:Person }", entailment=ENTAILMENT.RDFS)
print("alice is a Person (via rdfs:domain):     ", [str(r.x) for r in rows])

rows = g2.query("PREFIX ex: <http://example.org/> SELECT ?x WHERE { ?x a ex:Organization }", entailment=ENTAILMENT.RDFS)
print("Acme is an Organization (via rdfs:range):", [str(r.x) for r in rows])

rows = g2.query("PREFIX ex: <http://example.org/> SELECT ?x WHERE { ex:alice ex:hasRelative ?x }", entailment=ENTAILMENT.RDFS)
print("alice hasRelative bob (via subPropertyOf):", [str(r.x) for r in rows])

alice is a Person (via rdfs:domain):      ['http://example.org/alice']
Acme is an Organization (via rdfs:range): ['http://example.org/Acme']
alice hasRelative bob (via subPropertyOf): ['http://example.org/bob']


## 3. `entailment=ENTAILMENT['OWL-RDF-Based']` — OWL 2 RDF-Based Semantics

See the [Graph Inferencing guide](02b-graphs-inferencing.ipynb#3.-OWL-2-RL-reasoning) for what this regime covers (`owl:equivalentClass`/`owl:equivalentProperty`, property characteristics, `owl:sameAs`) and why it's the OWL 2 RL fragment rather than full OWL 2 RDF-Based Semantics. Applied here at query time, same as section 2 above.


### 3.1 `owl:equivalentClass` and `owl:equivalentProperty`

Applied at query time:


In [6]:
# Alice is a team lead is entailed from Alice is a manager, because team lead and manager are the same.
g_owl = StarLayerGraph()
g_owl.bind("ex", EX)
g_owl.parse(data='''
    @prefix ex: <http://example.org/> .
    @prefix owl: <http://www.w3.org/2002/07/owl#> .
    ex:Manager owl:equivalentClass ex:TeamLead .
    ex:alice a ex:Manager .
''', format="turtle12")

QUERY_OWL = "PREFIX ex: <http://example.org/> SELECT ?x WHERE { ?x a ex:TeamLead }"
print("entailment=ENTAILMENT.RDFS   (misses owl:equivalentClass):", [str(r.x) for r in g_owl.query(QUERY_OWL, entailment=ENTAILMENT.RDFS)])
print("entailment=ENTAILMENT['OWL-RDF-Based'] (catches it):                ", [str(r.x) for r in g_owl.query(QUERY_OWL, entailment=ENTAILMENT['OWL-RDF-Based'])])

entailment=ENTAILMENT.RDFS   (misses owl:equivalentClass): []
entailment=ENTAILMENT['OWL-RDF-Based'] (catches it):                 ['http://example.org/alice']


### 3.2 Property characteristics: `owl:TransitiveProperty`, `owl:SymmetricProperty`, `owl:inverseOf`

Applied at query time:


In [7]:
# SalesTeam is part of AcmeCorp is entailed via the transitive property.
# bob is friendOf alice is entailed via the symmetric property.
# dave is managedBy carol is entailed via the inverse property.
g_props = StarLayerGraph()
g_props.bind("ex", EX)
g_props.parse(data='''
    @prefix ex: <http://example.org/> .
    @prefix owl: <http://www.w3.org/2002/07/owl#> .

    ex:partOf a owl:TransitiveProperty .
    ex:SalesTeam ex:partOf ex:SalesDept .
    ex:SalesDept ex:partOf ex:AcmeCorp .

    ex:friendOf a owl:SymmetricProperty .
    ex:alice ex:friendOf ex:bob .

    ex:manages owl:inverseOf ex:managedBy .
    ex:carol ex:manages ex:dave .
''', format="turtle12")

rows = g_props.query("PREFIX ex: <http://example.org/> SELECT ?x WHERE { ex:SalesTeam ex:partOf ?x }", entailment=ENTAILMENT['OWL-RDF-Based'])
print("SalesTeam transitively part of :", sorted(str(r.x) for r in rows))

rows = g_props.query("PREFIX ex: <http://example.org/> SELECT ?x WHERE { ex:bob ex:friendOf ?x }", entailment=ENTAILMENT['OWL-RDF-Based'])
print("bob friendOf (symmetric)       :", [str(r.x) for r in rows])

rows = g_props.query("PREFIX ex: <http://example.org/> SELECT ?x WHERE { ex:dave ex:managedBy ?x }", entailment=ENTAILMENT['OWL-RDF-Based'])
print("dave managedBy (via inverseOf) :", [str(r.x) for r in rows])

SalesTeam transitively part of : ['http://example.org/AcmeCorp', 'http://example.org/SalesDept']
bob friendOf (symmetric)       : ['http://example.org/alice']
dave managedBy (via inverseOf) : ['http://example.org/carol']


### 3.3 `owl:sameAs` — congruence closure

Applied at query time:


In [8]:
#Alice Smith works at Acme, because Alice Smith and Alice Jones are the same entity.
g_same = StarLayerGraph()
g_same.bind("ex", EX)
g_same.parse(data='''
    @prefix ex: <http://example.org/> .
    @prefix owl: <http://www.w3.org/2002/07/owl#> .
    ex:aliceSmith owl:sameAs ex:aliceJones .
    ex:aliceJones ex:worksAt ex:Acme .
''', format="turtle12")

QUERY_SAME = "PREFIX ex: <http://example.org/> SELECT ?org WHERE { ex:aliceSmith ex:worksAt ?org }"
print("plain (no entailment):          ", [str(r.org) for r in g_same.query(QUERY_SAME)])
print("entailment=ENTAILMENT['OWL-RDF-Based'] (via sameAs):", [str(r.org) for r in g_same.query(QUERY_SAME, entailment=ENTAILMENT['OWL-RDF-Based'])])

plain (no entailment):           []
entailment=ENTAILMENT['OWL-RDF-Based'] (via sameAs): ['http://example.org/Acme']


### 3.4 Caching via `infer=`

Unlike `entailment=ENTAILMENT.RDFS`, `entailment=ENTAILMENT['OWL-RDF-Based']` requires a potentially time-consuming calculation of the entailed graph.  

By default (`infer="changed"`), `entailment=ENTAILMENT['OWL-RDF-Based']` will cache the entailed graph and reuse it for subsequent queries unless the underlying graph changes.  The other choices are `infer="cached"` which will force the re-use of the cached entailed graph regardless of changes and  `infer="always"` will force a re-compute of the entailed graph.  

In [9]:
g_owl.add((EX.carol, RDF.type, EX.Manager))
rows = g_owl.query(QUERY_OWL, entailment=ENTAILMENT['OWL-RDF-Based'])
print("default (changed), picks up carol:", sorted(str(r.x) for r in rows))

g_owl.add((EX.dave, RDF.type, EX.Manager))
stale = g_owl.query(QUERY_OWL, entailment=ENTAILMENT['OWL-RDF-Based'], infer="cached")
print("infer=\"cached\" (misses dave - stale on purpose):", sorted(str(r.x) for r in stale))

fresh = g_owl.query(QUERY_OWL, entailment=ENTAILMENT['OWL-RDF-Based'], infer="always")
print("infer=\"always\" (picks up dave):                 ", sorted(str(r.x) for r in fresh))

default (changed), picks up carol: ['http://example.org/alice', 'http://example.org/carol']
infer="cached" (misses dave - stale on purpose): ['http://example.org/alice', 'http://example.org/carol']
infer="always" (picks up dave):                  ['http://example.org/alice', 'http://example.org/carol', 'http://example.org/dave']


### 3.5 Detecting inconsistencies

`entailment=ENTAILMENT['OWL-RDF-Based']` inherits `owlrl`'s consistency checks (see the Inferencing guide's [section 3.4](02b-graphs-inferencing.ipynb#3.4-Detecting-inconsistencies)). When querying using `owl-rl` entailment, a result will be returned even if the underlying inference detects an inconsistency. Running a simple `ASK` query can detect the inconsistency:

In [10]:
ASK_INCONSISTENT = """
PREFIX err: <http://www.daml.org/2002/03/agents/agent-ont#>
ASK { ?e a err:ErrorMessage }
"""

# Dog and Cat are declared disjoint, but fido is asserted to be both
g_bad = StarLayerGraph()
g_bad.bind("ex", EX)
g_bad.parse(data='''
    @prefix ex: <http://example.org/> .
    @prefix owl: <http://www.w3.org/2002/07/owl#> .
    ex:Dog owl:disjointWith ex:Cat .
    ex:fido a ex:Dog, ex:Cat .
''', format="turtle12")

print("g_bad (genuine inconsistency):  ", bool(g_bad.query(ASK_INCONSISTENT, entailment=ENTAILMENT['OWL-RDF-Based'])))

# a normal query still returns a result - nothing is retracted, the inconsistency isn't surfaced here
rows = g_bad.query("PREFIX ex: <http://example.org/> SELECT ?c WHERE { ex:fido a ?c }", entailment=ENTAILMENT['OWL-RDF-Based'])
print("fido's classes (still returned despite the inconsistency):", sorted(str(r.c) for r in rows))

g_bad (genuine inconsistency):   True
fido's classes (still returned despite the inconsistency): ['http://example.org/Cat', 'http://example.org/Dog']
